In [3]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Postavke povezivanja (koristimo MySQL prema izvorima [3])
USER = 'root'
PASSWORD = '3k0p13!4'
HOST = 'localhost'
DB_NAME = 'fipu_srp_projekt'

# Kreiranje engine-a
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DB_NAME}")

# 2. SQL skripta za kreiranje Star sheme
sql_statements = [
    """
    CREATE TABLE IF NOT EXISTS dim_projekt (
        projekt_key INT AUTO_INCREMENT PRIMARY KEY,
        naziv_projekta VARCHAR(255)
    );
    """,
    """
    CREATE TABLE IF NOT EXISTS dim_tehnicar (
        tehnicar_key INT AUTO_INCREMENT PRIMARY KEY,
        ime_prezime VARCHAR(255)
    );
    """,
    """
    CREATE TABLE IF NOT EXISTS dim_prioritet_status (
        prioritet_status_key INT AUTO_INCREMENT PRIMARY KEY,
        razina_prioriteta VARCHAR(50),
        naziv_statusa VARCHAR(50)
    );
    """,
    """
    CREATE TABLE IF NOT EXISTS dim_vrijeme (
        vrijeme_key DATE PRIMARY KEY,
        dan INT,
        mjesec INT,
        godina INT,
        kvartal INT,
        dan_u_tjednu VARCHAR(20)
    );
    """,
    """
    CREATE TABLE IF NOT EXISTS fact_support_tickets (
        ticket_id INT PRIMARY KEY,
        projekt_key INT,
        reporter_key INT,
        assignee_key INT,
        prioritet_status_key INT,
        vrijeme_key DATE,
        vrijeme_rjesavanja_sati DECIMAL(10, 2),
        broj_komentara INT,
        CONSTRAINT fk_projekt FOREIGN KEY (projekt_key) REFERENCES dim_projekt(projekt_key),
        CONSTRAINT fk_reporter FOREIGN KEY (reporter_key) REFERENCES dim_tehnicar(tehnicar_key),
        CONSTRAINT fk_assignee FOREIGN KEY (assignee_key) REFERENCES dim_tehnicar(tehnicar_key),
        CONSTRAINT fk_status FOREIGN KEY (prioritet_status_key) REFERENCES dim_prioritet_status(prioritet_status_key),
        CONSTRAINT fk_vrijeme FOREIGN KEY (vrijeme_key) REFERENCES dim_vrijeme(vrijeme_key)
    );
    """
]

# 3. Izvršavanje upita
try:
    with engine.connect() as connection:
        # SQLAlchemy zahtijeva transakciju za DDL naredbe
        trans = connection.begin()
        for statement in sql_statements:
            connection.execute(text(statement))
        trans.commit()
        print("Sve tablice dimenzijskog modela su uspješno kreirane.")
except Exception as e:
    print(f"Došlo je do pogreške prilikom kreiranja tablica: {e}")

Sve tablice dimenzijskog modela su uspješno kreirane.
